# AAPL Stock Return — Statistical Analysis

**Author:** Lijin Gao  
**Program:** M.Sc. Financial Engineering — HEC Montréal  
**Data Source:** Yahoo Finance (`yfinance`), AAPL 2020–2025

---

## Objective

Apply core statistical concepts to real financial data (Apple stock):

1. Compute log returns and descriptive statistics
2. Visualize the return distribution vs. the normal distribution
3. Demonstrate the **Central Limit Theorem (CLT)** with resampling
4. Construct a **95% confidence interval** for the mean return
5. Run a **one-sided t-test** to check whether AAPL's mean return is significantly positive

## 1. Import Libraries & Download Data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import yfinance as yf

# Download AAPL data
raw = yf.download("AAPL", start="2020-01-01", end="2025-01-01", auto_adjust=True)

# Keep only closing price
prices = raw["Close"].squeeze()   # squeeze() converts single-column DataFrame to Series
prices.name = "AAPL"

print(f"{len(prices)} trading days downloaded")
print(f"Period: {prices.index[0].date()} to {prices.index[-1].date()}")
prices.tail()

## 2. Compute Log Returns

We use **log returns**: $r_t = \ln(P_t / P_{t-1})$

Log returns are preferred in finance because they are time-additive and approximately normally distributed for short horizons.

In [ ]:
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.name = "Log Return"

log_returns.tail()

## 3. Descriptive Statistics

Key statistics of the daily log return distribution.

In [ ]:
mean   = log_returns.mean()
std    = log_returns.std()
skew   = log_returns.skew()
kurt   = log_returns.kurtosis()   # excess kurtosis (normal = 0)

print("Descriptive Statistics — AAPL Daily Log Returns")
print("-" * 48)
print(f"  Observations : {len(log_returns)}")
print(f"  Mean         : {mean:.6f}  ({mean * 252:.2%} annualized)")
print(f"  Std Dev      : {std:.6f}  ({std * np.sqrt(252):.2%} annualized)")
print(f"  Skewness     : {skew:.4f}")
print(f"  Excess Kurt. : {kurt:.4f}")
print()
print("Note: positive excess kurtosis means fat tails (more extreme events than normal).")

## 4. Return Distribution vs. Normal Curve

We overlay a fitted normal distribution on the histogram to visually assess normality.  
If the tails of the histogram are heavier than the curve → **fat tails**, a well-known feature of financial returns.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Histogram (density=True so it matches the normal PDF scale)
ax.hist(log_returns, bins=60, density=True,
        color="steelblue", edgecolor="white", linewidth=0.3,
        alpha=0.8, label="Observed returns")

# Fitted normal curve
x = np.linspace(log_returns.min(), log_returns.max(), 300)
ax.plot(x, norm.pdf(x, mean, std), color="tomato", lw=2, label="Normal fit")

# Mean line
ax.axvline(mean, color="black", linestyle="--", lw=1.2, label=f"Mean = {mean:.4f}")

ax.set_title("AAPL Daily Log Return Distribution (2020–2025)")
ax.set_xlabel("Log Return")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig("plots/return_distribution.png", bbox_inches="tight")
plt.show()

## 5. Central Limit Theorem (CLT) Demonstration

The CLT states: even if individual observations are **not** normally distributed, the **distribution of sample means** converges to a normal distribution as sample size grows.

We verify this by drawing 1,000 random samples of size 30 and plotting the distribution of their means.

In [ ]:
np.random.seed(42)
SAMPLE_SIZE = 30
N_SAMPLES   = 1000

sample_means = []
for _ in range(N_SAMPLES):
    sample = log_returns.sample(SAMPLE_SIZE)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(sample_means, bins=50, density=True,
        color="mediumseagreen", edgecolor="white", linewidth=0.3,
        alpha=0.85, label=f"Sample means (n={SAMPLE_SIZE})")

# Theoretical normal distribution of sample means: N(μ, σ²/n)
theoretical_std = std / np.sqrt(SAMPLE_SIZE)
x = np.linspace(sample_means.min(), sample_means.max(), 300)
ax.plot(x, norm.pdf(x, mean, theoretical_std),
        color="tomato", lw=2, label="Theoretical normal (CLT)")

ax.set_title(f"CLT Demo: Distribution of {N_SAMPLES} Sample Means (n={SAMPLE_SIZE})")
ax.set_xlabel("Sample Mean")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig("plots/clt_demo.png", bbox_inches="tight")
plt.show()

print("The histogram closely follows the theoretical normal curve — CLT confirmed.")

## 6. 95% Confidence Interval for the Mean Return

A 95% CI means: if we repeated this study many times, 95% of the intervals constructed would contain the true population mean.

Formula: $\bar{x} \pm z_{0.975} \cdot \dfrac{s}{\sqrt{n}}$

In [ ]:
n  = len(log_returns)
z  = norm.ppf(0.975)           # 1.96 for 95% CI
se = std / np.sqrt(n)          # standard error

lower = mean - z * se
upper = mean + z * se

print("95% Confidence Interval for Mean Daily Log Return")
print("-" * 50)
print(f"  Sample mean : {mean:.6f}")
print(f"  Std error   : {se:.6f}")
print(f"  z (97.5%)   : {z:.4f}")
print(f"  Lower bound : {lower:.6f}  ({lower:.4%})")
print(f"  Upper bound : {upper:.6f}  ({upper:.4%})")
print()
print(f"Interpretation: We are 95% confident the true mean daily return")
print(f"lies between {lower:.4%} and {upper:.4%}.")
print()
if lower > 0:
    print("The interval is entirely above 0 → the mean return is statistically positive at the 95% level.")
else:
    print("The interval contains 0 → we cannot conclude the mean return is significantly positive")
    print("from the CI alone. Use the t-test below for a formal significance test.")

## 7. Hypothesis Test — Is the Mean Return Significantly Positive?

**Hypotheses (one-sided t-test):**

- $H_0$: $\mu \leq 0$ — the mean daily return is zero or negative  
- $H_1$: $\mu > 0$ — the mean daily return is significantly positive

We use a **one-sample t-test** at significance level $\alpha = 5\%$.

In [ ]:
t_stat, p_two_tail = stats.ttest_1samp(log_returns, popmean=0)

# One-sided p-value (we only care about H1: μ > 0)
p_one_tail = p_two_tail / 2

alpha = 0.05

print("One-Sided t-Test Results")
print("-" * 40)
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value (1-tail) : {p_one_tail:.6f}")
print(f"  Significance level (α) : {alpha}")
print()

if p_one_tail < alpha and t_stat > 0:
    print("Decision: Reject H₀")
    print("The mean daily return is statistically significantly positive (α = 5%).")
else:
    print("Decision: Fail to reject H₀")
    print("Not enough evidence to conclude the mean return is positive.")

## 8. Conclusion

| Analysis | Result |
|----------|--------|
| Mean daily log return | ~0.09% (≈ 23% annualized) |
| Annualized volatility | ~30% |
| Excess kurtosis | > 0 → fat tails (non-normal) |
| CLT demonstration | Sample means converge to normal ✓ |
| 95% CI | Contains 0 → inconclusive on its own; use t-test for formal test |
| Hypothesis test | Reject H₀ — return is significantly positive |

---

**Key takeaway:** AAPL delivered a statistically significant positive return over 2020–2025. However, the fat tails observed in the return distribution highlight that simple normality assumptions underestimate tail risk — an important consideration for risk management.

> *This project is for educational purposes only and does not constitute investment advice.*